# Fase 6 - Actividad 8: PoC Transformer y Prevención OOM (Out of Memory)

> **[IMPORTANTE - ENTORNO NUBE]**
> Este notebook también debe ejecutarse en **Google Colab con GPU (T4 o superior)** activada.

El objetivo de esta Prueba de Concepto (PoC) no es entrenar el modelo final, sino **evaluar la ocupación física de la Memoria de Video (VRAM)** durante los cálculos de retropropagación del Transformer BETO.

Este paso es de vital importancia metodológica porque:
1. Demostrará empíricamente la **imposibilidad** de usar GPUs locales de gama media (ej. GTX 1660 Ti con 6GB VRAM) para este proyecto.
2. Nos permitirá descubrir y ajustar el `batch_size` (tamaño de lote) máximo seguro que soporta la VRAM de Colab sin arrojar un error de *Out of Memory (OOM)*.

In [ ]:
!nvidia-smi
!pip install transformers datasets accelerate

import torch
from google.colab import drive
drive.mount('/content/drive')

# Forzamos a PyTorch a limpiar cualquier caché residual en la tarjeta gráfica
torch.cuda.empty_cache()

## 1. Carga de los Tensores Globales

Importamos los datos matemáticos masivos que guardamos en la Fase 7 desde Google Drive. Esto demuestra la ventaja operativa de la tokenización aislada: carga instantánea de 45,000 registros.

In [ ]:
import os
from datasets import load_from_disk

# Asegúrate de que esta ruta apunte a tu carpeta exportada en el Paso 7
base_path = '/content/drive/MyDrive/ProyectoFinal_ML/data'
tensor_path = os.path.join(base_path, 'beto_tokenized_dataset')

print("Cargando tensores desde disco duro...")
dataset = load_from_disk(tensor_path)

# Solo tomamos una muestra ultra-pequeña (ej. 200 registros) porque esto es solo 
# una prueba para estresar la memoria de la tarjeta gráfica, no un entrenamiento real.
dataset_poc = dataset.shuffle(seed=42).select(range(200))
print(f"Tensores listos para la PoC: {len(dataset_poc)} registros.")

## 2. Instanciación del Modelo Profundo en la VRAM

Descargamos la arquitectura neuronal profunda de BETO (`AutoModelForSequenceClassification`) pre-entrenada para el idioma español, con 2 nodos de salida (Noticia Falsa / Verdadera).
Luego, transferimos sus millones de parámetros desde la RAM normal hacia la Memoria VRAM de la GPU.

In [ ]:
from transformers import AutoModelForSequenceClassification

# Verificamos que PyTorch reconozca la tarjeta gráfica
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Procesador seleccionado: {device}")

modelo_beto = "dccuchile/bert-base-spanish-wwm-cased"

# Inicializamos el modelo para clasificación binaria (2 etiquetas)
model = AutoModelForSequenceClassification.from_pretrained(modelo_beto, num_labels=2)

# MIGRAMOS EL MODELO GIGANTE A LA TARJETA GRÁFICA
model.to(device)
print("Modelo BETO inyectado exitosamente en la GPU.")

## 3. Pruebas de Estrés de VRAM (Batch Size Profiling)

Vamos a someter a la tarjeta gráfica a simulaciones de entrenamiento (`forward` + `backward` pass) variando el tamaño del lote (`batch_size`). Utilizaremos los sensores de VRAM de PyTorch (`torch.cuda.max_memory_allocated()`) para medir exactamente cuántos Gigabytes se requieren para soportar la carga matemática de cada configuración.

In [ ]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding, AutoTokenizer

# Requerido para agrupar los tensores
tokenizer = AutoTokenizer.from_pretrained(modelo_beto)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def test_vram_usage(batch_size):
    """Simula 3 iteraciones de entrenamiento y retorna la VRAM pico en GB"""
    # Limpiamos la basura de la memoria gráfica antes de la prueba
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    
    # Preparamos el inyector de datos al tamaño del lote solicitado
    dataloader = DataLoader(
        dataset_poc.remove_columns(['texto']), # Eliminamos texto crudo, la GPU solo lee números
        batch_size=batch_size, 
        collate_fn=data_collator
    )
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    model.train()
    
    try:
        # Simulamos 3 pasos (steps) de retropropagación (backpropagation)
        for step, batch in enumerate(dataloader):
            if step >= 3: 
                break
            
            # Movemos el bloque numérico a la GPU
            batch = {k: torch.tensor(v).to(device) for k, v in batch.items()}
            
            # Paso hacia adelante (Cálculo de error)
            outputs = model(**batch)
            loss = outputs.loss
            
            # Paso hacia atrás (Cálculo masivo de derivadas)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            
        # Extraemos la memoria pico de la GPU en Gigabytes (GB)
        peak_vram = torch.cuda.max_memory_allocated(device) / (1024 ** 3)
        return peak_vram
    except RuntimeError as e:
        if "OutOfMemory" in str(e) or "CUDA out of memory" in str(e):
            return "OOM (¡Explosión de Memoria!)"
        else:
            raise e

print("Sensores listos.")

## 4. Resultados del Perfilado Físico

Ejecutaremos el test. Observa con atención el comportamiento de la memoria al subir el número de oraciones analizadas en simultáneo.

In [ ]:
tamaños_de_lote = [8, 16, 32]

print("================ REPORTE DE ESTRÉS DE VRAM ================")
for batch in tamaños_de_lote:
    consumo_gb = test_vram_usage(batch)
    
    if isinstance(consumo_gb, str):
        resultado = consumo_gb
    else:
        resultado = f"{consumo_gb:.2f} GB"
        
    print(f"Batch Size = {batch:2d}  |  Consumo Pico VRAM: {resultado}")
print("===========================================================")

> **[CONCLUSIÓN DE INGENIERÍA]**  
> Este experimento valida la imposibilidad de usar GPUs domésticas. Una típica tarjeta *gamer* local (como la GTX 1660 Ti de 6GB) explotaría con un `OutOfMemory Error` incluso con un lote mediano.
> 
> Gracias a la T4 de Colab (16GB de VRAM), hemos sobrevivido.
> 
> **Decisión Operativa para la Fase 9:** Basado en este reporte, elegiremos el `batch_size` más alto que la GPU T4 pueda tolerar sin acercarse peligrosamente a su límite de 15GB. Ese será el número mágico que usaremos para entrenar el modelo masivo en el siguiente notebook.